# Minimization

This assumes you have already solvated your protein using the previous script, and have a file of the form `{id}_solvated.pdb`

Goals for this notebook.
1. Define simulation parameters: forcefield, timestep, temperature, etc..
2. Minimize the structure, with contraints on heavy atoms and any ligands
3. Save the minimized structure

Note: This is the last stage you can run in a reasonable time on a CPU.  For heating, equilibration, and production dynamics, a powerful GPU is highly recommended.

In [ ]:
from pathlib import Path

import openmm as mm
import openmm.app as app
import openmm.unit as unit

import py3Dmol

from sys import stdout


## Setup Protein in a water box

To mimic the cellular environment, we usually immerse the protein in a box of water molecules. If you have ligands or non-standard residues, the setup is more complicated.  Master the protein setup first before moving on to those cases.


In [ ]:
id = '4trn'
Path('equilibration').mkdir(exist_ok=True)

# ========== Run Parameters ===========
temperature = 300 * unit.kelvin
friction = 1 / unit.picosecond
timestep = 2 * unit.femtoseconds
integrator = mm.LangevinMiddleIntegrator(temperature, friction, timestep)
platform = mm.Platform.getPlatformByName('CPU')  # nvidia gpu - 'CUDA' apple silicon (M1/M2.. etc) - 'OpenCl'
# ========== Run Parameters ===========

print("Loading Coordinates...")
pdb = app.PDBFile(f"./pdbs/{id}_solvated.pdb")
modeller = app.Modeller(pdb.topology, pdb.positions)

print("Creating System...")
forcefield = app.ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,              
    nonbondedCutoff=1.0 * unit.nanometer, 
    constraints=app.HBonds                
)

print("Creating and saving simulation object...")
simulation = app.Simulation(modeller.topology, system, integrator, platform)
simulation.context.setPositions(modeller.positions)

# Saving the molecular mechanics system
with open(f"./equilibration/{id}_system.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(system))

PE = simulation.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)

print(f"Number of atoms: {system.getNumParticles():,}")
print(f"Initial Potential Energy = {PE :.4f} kJ/mol ")
print("Simulation object ready...  Ready for minimization")

## Restrained Minimization

Because the initial energy is usually quite high, we generally place a restraint on the protein backbone, keeping it roughly in the x-ray pose, while allowing all the side chains, waters, and hydrogen atoms to relax.  The restraint takes the form of a harmonic potential ($ \frac{1}{2} k (r - r_0)^2$)

To apply these contraints, there's several helper functions below.

In [ ]:
# Restraint definitions
def create_restraint():
    force_function = "0.5*k*((x-x0)^2 + (y-y0)^2 + (z-z0)^2)"
    force = mm.CustomExternalForce( force_function )
    force.addPerParticleParameter("x0")
    force.addPerParticleParameter("y0")
    force.addPerParticleParameter("z0")
    force.addPerParticleParameter("k")
    return force

def backbone_restraint( simulation, force, k):
    positions = simulation.context.getState(getPositions=True).getPositions(asNumpy=True)
    for atom in simulation.topology.atoms():
        if atom.name in ["CA", "N", "C"]:
            pos = positions[atom.index].value_in_unit(unit.nanometer)
            force.addParticle(atom.index, [*pos, k])
    return

def residue_restraint( simulation, force, residue_names, k):
    positions = simulation.context.getState(getPositions=True).getPositions(asNumpy=True)
    for atom in simulation.topology.atoms():
        if ( atom.residue.name in residue_names ) and atom.element.symbol != "H":
            pos = positions[atom.index].value_in_unit(unit.nanometer)
            force.addParticle(atom.index, [*pos, k])
    return



## Minimization

In [ ]:
# ====== Restraints ON =======
k = 1000 * unit.kilojoule_per_mole / unit.nanometer**2
restraint_force = create_restraint()
backbone_restraint(simulation, restraint_force, k)
restraint_id = system.addForce(restraint_force)
simulation.context.reinitialize(preserveState=True)

# ======= Minimize ========
simulation.minimizeEnergy(maxIterations=1000)

# ====== Restraints OFF =======
system.removeForce(restraint_id) 
simulation.context.reinitialize(preserveState=True)

PE = simulation.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
print(f"Minimized Potential Energy = {PE :.4f} kJ/mol ")

# ====== Saving System State and Coordinates =======
positions = simulation.context.getState(getPositions=True).getPositions()
with open(f"./equilibration/{id}_min.pdb", "w") as f:
    app.PDBFile.writeFile(simulation.topology, positions, f)

simulation.saveState(f"./equilibration/{id}_min.xml")

print(f"Minimization complete and structure saved as {id}_min.pdb!")

## Visualization of minimized structure

I'd encourage you to open both the original un-minimized structure, `{id}_solvated.pdb` and the minimized structure, `{id}_min.pdb`, in pymol, zoom in on key residues and look for any changes.  They should be subtle since we are restraining the backbone, but there will be noticable shifts in side chains and hydrogens as they adopt lower energy configurations.